In [0]:
dbutils.widgets.text("catalog","fk_1361")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql import functions as F

In [0]:
df = spark.read.table(f"{catalog}.bronze.orders_bronze")

In [0]:
df.display()

In [0]:
df_customer = spark.read.table(f"{catalog}.silver.customers_silver")
df_products = spark.read.table(f"{catalog}.silver.products_silver")

In [0]:
df = df.join(df_customer, "customer_id","left_semi")

In [0]:
df_orders = df.withColumns({
    "order_id": F.trim(F.upper("order_id")),
    "customer_id": F.trim(F.upper(F.col("customer_id"))),
    "store_id" : F.trim(F.upper(F.col("store_id"))),
    "order_ts" : F.coalesce(F.try_to_timestamp("order_ts", F.lit("yyyy-MM-dd HH:mm:ss")), F.try_to_timestamp("order_ts", F.lit("yyyy-MM-dd'T'HH:mm:ss"))), 
    # "order_date" : F.to_date(F.col("order_ts"), "yyyy-MM-dd"),
    "order_status" : F.trim(F.upper(F.col("order_status"))),
    "payment_method" : F.trim(F.upper(F.col("payment.method"))),
    "promised_mins" : F.col("delivery.promised_mins").cast("int"),
    "actual_mins" : F.col("delivery.actual_mins").cast("int"),
    "delivery_fee" : F.col("delivery.delivery_fee").cast("double"),
    "update_at" : F.try_to_timestamp("updated_at", F.lit("yyyy-MM-dd HH:mm:ss"))
    }) \
    .select("order_id","customer_id","store_id","order_ts","order_status","payment_method","promised_mins","actual_mins","delivery_fee","update_at")


In [0]:
df_orders.display()

In [0]:
df_orders.write.mode("overwrite").saveAsTable(f"{catalog}.silver.orders_silver")

In [0]:
df_expode = df.withColumn("items", F.explode(F.from_json(F.col("items"), "ARRAY<STRUCT<product_id: STRING, qty: INT, unit_price: DOUBLE, discount: DOUBLE>>")))

In [0]:
df_order_items = df_expode.withColumns({
    "order_id" : F.trim(F.upper(F.col("order_id"))),
    "product_id" : F.col("items.product_id"),
    "quantity" : F.col("items.qty").cast("int"),
    "unit_price" : F.col("items.unit_price").cast("double"),
    "discount_amou nt" : F.coalesce(F.col("items.discount").cast("double"), F.lit(0)),
    "gross_amount" : F.col("unit_price") * F.col("quantity"),
    "net_amount" : F.col("gross_amount") - F.col("discount_amount")
}) \
    .filter((F.col("quantity") > 0) & (F.col("unit_price") > 0)) \
    .dropDuplicates() \
    .dropna(subset = ["customer_id","store_id","order_ts","items","product_id"]) \
    .select("order_id","product_id", "quantity","unit_price","discount_amount","gross_amount","net_amount")

In [0]:
df_order_items.write.mode("overwrite").saveAsTable(f"{catalog}.silver.order_items_silver")

In [0]:
%sql
select * from 

# New cleaning orders

In [0]:
bronze = spark.read.table(f"{catalog}.bronze.orders_bronze")

b_cols = bronze.columns
def sid(c):
    d = F.upper(F.trim(c)).when(d == "", F.lit(None)).otherwise(d)
    return F

In [0]:
TS_RE  = r"^\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}$"   # S5
UPD_RE = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"      # S6

In [0]:
bronze = spark.table(f"{catalog}.bronze.orders_bronze")
b_cols = bronze.columns

def sid(c):                                   # S1
    t = F.upper(F.trim(c))
    return F.when(t == "", F.lit(None)).otherwise(t)

TS_RE  = r"^\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}$"   # S5
UPD_RE = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"      # S6

status = F.upper(F.trim(F.col("order_status")))
chan   = F.upper(F.trim(F.col("channel"))) if "channel" in b_cols else F.lit(None)

std = (bronze
  .select(
    sid(F.col("order_id")).alias("order_id"),
    sid(F.col("customer_id")).alias("customer_id"),
    sid(F.col("store_id")).alias("store_id"),
    F.col("order_ts").cast("string").alias("order_ts_raw"),
    F.col("updated_at").cast("string").alias("updated_at_raw"),
    F.when(status == "CANCELED", "CANCELLED").otherwise(status).alias("order_status"),   # S2
    F.upper(F.trim(F.col("payment.method"))).alias("payment_method"),                    # S3
    F.coalesce(F.when(chan == "", F.lit(None)).otherwise(chan), F.lit("UNKNOWN")).alias("channel"),  # S4
    F.expr("try_cast(try_cast(delivery.promised_mins as decimal(18,2)) as int)").alias("promised_mins"),
    F.expr("try_cast(try_cast(delivery.actual_mins as decimal(18,2)) as int)").alias("actual_mins"),
    F.expr("try_cast(cast(delivery.delivery_fee as string) as decimal(10,2))").alias("delivery_fee"),
    F.col("items"),
    F.to_json(F.struct(*[F.col(c) for c in b_cols])).alias("raw_record"))
  .withColumn("order_ts", F.when(F.col("order_ts_raw").rlike(TS_RE),
        F.expr("try_to_timestamp(replace(order_ts_raw,'T',' '),'yyyy-MM-dd HH:mm:ss')")))
  .withColumn("order_date", F.when(F.col("order_ts").isNotNull(),
        F.to_date(F.substring("order_ts_raw", 1, 10))))
  .withColumn("updated_at", F.when(F.col("updated_at_raw").rlike(UPD_RE),
        F.expr("try_to_timestamp(updated_at_raw,'yyyy-MM-dd HH:mm:ss')"))))

std.display()


In [0]:
from pyspark.sql import Window
w = Window.partitionBy("order_id").orderBy(
    F.col("updated_at").desc_nulls_last())

latest = (std.where("order_id IS NOT NULL")
             .withColumn("rn", F.row_number().over(w))
             .where("rn = 1").drop("rn"))

latest.display()

In [0]:
"" == NULL

In [0]:
F.upper(F.trim("")).display()

In [0]:
sample = spark.createDataFrame([("",), ("  ",), ("123",)], ["value"])

In [0]:
sample.display()

In [0]:
sample.select(F.upper(F.trim("value"))).dropna().display()